# Variable Elimination

## Learning goals

By the end of this notebook, we should be able to:

1. Explain why variable elimination is needed.
2. Identify query, evidence, and hidden variables.
3. Understand the expression: $$P(Q|e)=\alpha \sum_H \prod_i \phi_i$$
4. Eliminate one hidden variable using factor multiplication and marginalization.
5. Implement exact inference in a discrete Bayesian network.

## Main Idea

A Bayesian network stores a joint probability distribution compactly through local conditional probability tables. 
$$P(B,E,A,J,M)=P(B)P(E)P(A|B,E)P(J|A)P(M|A)$$

Suppose we want: $$P(B\mid J=\text{True},M=\text{True})$$

We do **not** need the complete joint table. We only need the part of the computation relevant to this query.

Variable elimination:

- fixes the evidence,
- combines factors that contain a hidden variable,
- sums out that hidden variable,
- repeats until only the query variable remains,
- normalizes the result.

## Query, evidence, and hidden variables

For the query

$$P(B\mid J=\text{True},M=\text{True})$$

we have:

- **Query variable:** $(B)$
- **Evidence variables:** $(J)$ and $(M)$
- **Hidden variables:** $(E)$ and $(A)$

In general,

$$H=\mathcal{V}\setminus(Q\cup E)$$

where:

- $(mathcal{V})$ is the set of all variables,
- $(Q)$ is the set of query variables,
- $(E)$ is the set of evidence variables,
- $(H)$ is the set of hidden variables.

## The operation performed for one hidden variable

Suppose we want to eliminate $(Z)$.

1. Find every factor containing $(Z)$.
2. Multiply those factors.
3. Sum out $(Z)$.
4. Put the resulting factor back.

Symbolically, if the factors containing $(Z)$ are

$$
\phi_1(Z,X),\qquad
\phi_2(Z,Y),\qquad
\phi_3(Z),
$$

then they are replaced by

$$\tau(X,Y)=
\sum_Z
\phi_1(Z,X)\phi_2(Z,Y)\phi_3(Z).
$$

The information contributed by \(Z\) is preserved, but \(Z\) itself disappears from the new factor.

## Tiny mental example

Consider the chain

$$
A\rightarrow B\rightarrow C.
$$

The joint distribution is

$$
P(A,B,C)=P(A)P(B\mid A)P(C\mid B).
$$

Suppose \(A=a\) is observed and we want $P(C\mid A=a)$.

The variable \(B\) is hidden:

$$
P(C\mid A=a)
\propto
\sum_B P(B\mid A=a)P(C\mid B).
$$

We first combine the factors involving \(B\), then sum over all values of \(B\).

After this operation, \(B\) is gone, but its influence on \(C\) remains.

## Implementation

The variable-elimination algorithm needs only four factor operations:

- `restrict(variable, value)`
- factor multiplication
- `sum_out(variable)`
- `normalize()`

The compact `Factor` class below makes this notebook runnable by itself. If you already have your own `Factor` class, you may replace this class while keeping the variable-elimination functions unchanged.


In [16]:
from __future__ import annotations

from dataclasses import dataclass
from functools import reduce
from typing import Any, Iterable, Mapping, Sequence

import numpy as np

In [17]:
@dataclass(frozen=True)
class Factor:
    """
    A discrete factor represented by:
        variables: ordered variable names
        cardinalities: number of states for every variable
        values: NumPy array whose axes follow `variables`
    """
    variables: tuple[str, ...]
    cardinalities: Mapping[str, int]
    values: np.ndarray

    def __post_init__(self) -> None:
        variables = tuple(self.variables)
        values = np.asarray(self.values, dtype=float)

        if len(set(variables)) != len(variables):
            raise ValueError("Factor variables must be unique.")

        expected_shape = tuple(self.cardinalities[v] for v in variables)
        if values.shape != expected_shape:
            raise ValueError(
                f"Expected values shape {expected_shape}, got {values.shape}."
            )

        object.__setattr__(self, "variables", variables)
        object.__setattr__(self, "values", values)

    def contains(self, variable: str) -> bool:
        return variable in self.variables

    def restrict(self, variable: str, value: int) -> "Factor":
        """Condition the factor on variable=value."""
        if variable not in self.variables:
            return self

        cardinality = self.cardinalities[variable]
        if not 0 <= value < cardinality:
            raise ValueError(
                f"{variable} has states 0,...,{cardinality - 1}; got {value}."
            )

        axis = self.variables.index(variable)
        restricted_values = np.take(self.values, indices=value, axis=axis)
        new_variables = tuple(v for v in self.variables if v != variable)

        return Factor(
            variables=new_variables,
            cardinalities=self.cardinalities,
            values=restricted_values,
        )

    def sum_out(self, variable: str) -> "Factor":
        """Marginalize a variable from the factor."""
        if variable not in self.variables:
            return self

        axis = self.variables.index(variable)
        marginalized_values = self.values.sum(axis=axis)
        new_variables = tuple(v for v in self.variables if v != variable)

        return Factor(
            variables=new_variables,
            cardinalities=self.cardinalities,
            values=marginalized_values,
        )

    def normalize(self) -> "Factor":
        """Scale all entries so that they sum to one."""
        total = self.values.sum()
        if total <= 0:
            raise ValueError("Cannot normalize a factor with non-positive mass.")

        return Factor(
            variables=self.variables,
            cardinalities=self.cardinalities,
            values=self.values / total,
        )

    def multiply(self, other: "Factor") -> "Factor":
        """Multiply two factors while aligning common variables."""
        for variable in set(self.variables) & set(other.variables):
            if self.cardinalities[variable] != other.cardinalities[variable]:
                raise ValueError(
                    f"Incompatible cardinality for variable {variable!r}."
                )

        result_variables = self.variables + tuple(
            v for v in other.variables if v not in self.variables
        )

        def align(factor: "Factor") -> np.ndarray:
            # Reorder existing axes in the order they appear in result_variables.
            existing_order = [v for v in result_variables if v in factor.variables]
            permutation = [factor.variables.index(v) for v in existing_order]
            aligned = (
                np.transpose(factor.values, axes=permutation)
                if permutation
                else factor.values
            )

            # Insert singleton dimensions for variables absent from the factor.
            shape = [
                factor.cardinalities[v] if v in factor.variables else 1
                for v in result_variables
            ]
            return aligned.reshape(shape)

        result_values = align(self) * align(other)
        merged_cardinalities = dict(self.cardinalities)
        merged_cardinalities.update(other.cardinalities)

        return Factor(
            variables=result_variables,
            cardinalities=merged_cardinalities,
            values=result_values,
        )

    def __mul__(self, other: "Factor") -> "Factor":
        return self.multiply(other)

    def __repr__(self) -> str:
        return (
            f"Factor(variables={self.variables}, "
            f"shape={self.values.shape})\n{self.values}"
        )


## Helper 1: multiply a collection of factors

Factor multiplication is associative, so we can multiply the factors sequentially.

$$\phi_1\phi_2\phi_3=
(\phi_1\phi_2)\phi_3.
$$

In [18]:
def multiply_all(factors: Sequence[Factor]) -> Factor:
    if not factors:
        raise ValueError("At least one factor is required.")

    return reduce(lambda left, right: left * right, factors)

## Helper 2: apply evidence

Evidence means that a variable is observed.

If $(J=\text{True})$, rows corresponding to $(J=\text{False})$ are no longer possible. Therefore, every factor containing $(J)$ is restricted to the observed value.

In this notebook, binary states are encoded as:

- `0`: False
- `1`: True


In [19]:
def apply_evidence(
    factors: Sequence[Factor],
    evidence: Mapping[str, int],
) -> list[Factor]:
    restricted_factors: list[Factor] = []

    for factor in factors:
        updated = factor
        for variable, value in evidence.items():
            if updated.contains(variable):
                updated = updated.restrict(variable, value)
        restricted_factors.append(updated)

    return restricted_factors

## Helper 3: eliminate one variable

For a hidden variable \(Z\):

- separate factors into those that contain \(Z\) and those that do not,
- multiply factors containing \(Z\),
- sum out \(Z\),
- return the untouched factors together with the new factor.

In [20]:
def eliminate_variable(
    factors: Sequence[Factor],
    variable: str,
) -> list[Factor]:
    related = [factor for factor in factors if factor.contains(variable)]
    unrelated = [factor for factor in factors if not factor.contains(variable)]

    if not related:
        return list(factors)

    product = multiply_all(related)
    reduced = product.sum_out(variable)

    return unrelated + [reduced]

## Main algorithm

The final result is

$$P(Q\mid e)=
\alpha
\sum_H
\prod_i \phi_i,
$$

where $\alpha$ is the normalization constant.

The algorithm below performs exactly this expression, but it eliminates hidden variables one at a time rather than constructing the complete joint distribution.

In [21]:
def variable_elimination(
    factors: Sequence[Factor],
    query_variables: Sequence[str],
    evidence: Mapping[str, int] | None = None,
    elimination_order: Sequence[str] | None = None,
) -> Factor:
    evidence = dict(evidence or {})
    query_variables = tuple(query_variables)

    if not query_variables:
        raise ValueError("At least one query variable is required.")

    overlap = set(query_variables) & set(evidence)
    if overlap:
        raise ValueError(
            f"Query variables cannot also be evidence variables: {sorted(overlap)}"
        )

    all_variables = {
        variable
        for factor in factors
        for variable in factor.variables
    }

    unknown_query = set(query_variables) - all_variables
    unknown_evidence = set(evidence) - all_variables

    if unknown_query:
        raise ValueError(f"Unknown query variables: {sorted(unknown_query)}")
    if unknown_evidence:
        raise ValueError(f"Unknown evidence variables: {sorted(unknown_evidence)}")

    hidden_variables = (
        all_variables
        - set(query_variables)
        - set(evidence)
    )

    if elimination_order is None:
        elimination_order = sorted(hidden_variables)
    else:
        elimination_order = tuple(elimination_order)
        if set(elimination_order) != hidden_variables:
            raise ValueError(
                "The elimination order must contain every hidden variable "
                "exactly once."
            )

    working_factors = apply_evidence(factors, evidence)

    for variable in elimination_order:
        working_factors = eliminate_variable(
            working_factors,
            variable,
        )

    result = multiply_all(working_factors)

    # Defensive cleanup: only query variables should remain.
    for variable in tuple(result.variables):
        if variable not in query_variables:
            result = result.sum_out(variable)

    # Reorder axes to match the requested query order.
    if result.variables != query_variables:
        permutation = [result.variables.index(v) for v in query_variables]
        result = Factor(
            variables=query_variables,
            cardinalities=result.cardinalities,
            values=np.transpose(result.values, axes=permutation),
        )

    return result.normalize()


# Example 1: $(A\rightarrow B\rightarrow C)$

We will query

$$
P(C\mid A=\text{True}).
$$

The hidden variable is $B$.

In [22]:
cardinalities = {"A": 2, "B": 2, "C": 2}

# P(A)
phi_A = Factor(
    variables=("A",),
    cardinalities=cardinalities,
    values=np.array([0.6, 0.4]),
)

# P(B | A)
# Axis order: A, B
phi_B_given_A = Factor(
    variables=("A", "B"),
    cardinalities=cardinalities,
    values=np.array([
        [0.8, 0.2],  # A=False
        [0.3, 0.7],  # A=True
    ]),
)

# P(C | B)
# Axis order: B, C
phi_C_given_B = Factor(
    variables=("B", "C"),
    cardinalities=cardinalities,
    values=np.array([
        [0.9, 0.1],  # B=False
        [0.2, 0.8],  # B=True
    ]),
)

chain_factors = [
    phi_A,
    phi_B_given_A,
    phi_C_given_B,
]


In [23]:
result = variable_elimination(
    factors=chain_factors,
    query_variables=["C"],
    evidence={"A": 1},
    elimination_order=["B"],
)

result


Factor(variables=('C',), shape=(2,))
[0.41 0.59]

### Manual check

Since $$(A=\text{True})$$,

$$P(B=\text{False}\mid A=\text{True})=0.3$$

and

$$P(B=\text{True}\mid A=\text{True})=0.7$$

Therefore,

$$P(C=\text{True}\mid A=\text{True})=
0.3(0.1)+0.7(0.8)=
0.59.
$$

The expected output is therefore approximately

```text
C=False: 0.41
C=True : 0.59
```

# Example 2: Alarm network

The network is

$$
B\rightarrow A\leftarrow E,
\qquad
A\rightarrow J,
\qquad
A\rightarrow M.
$$

We will compute

$$
P(B\mid J=\text{True},M=\text{True}).
$$

The hidden variables are $E$ and $A$.


In [24]:
alarm_cardinalities = {
    "Burglary": 2,
    "Earthquake": 2,
    "Alarm": 2,
    "JohnCalls": 2,
    "MaryCalls": 2,
}

phi_B = Factor(
    ("Burglary",),
    alarm_cardinalities,
    np.array([0.999, 0.001]),
)

phi_E = Factor(
    ("Earthquake",),
    alarm_cardinalities,
    np.array([0.998, 0.002]),
)

# P(Alarm | Burglary, Earthquake)
# Axis order: Burglary, Earthquake, Alarm
phi_A = Factor(
    ("Burglary", "Earthquake", "Alarm"),
    alarm_cardinalities,
    np.array([
        [
            [0.999, 0.001],  # B=0, E=0
            [0.710, 0.290],  # B=0, E=1
        ],
        [
            [0.060, 0.940],  # B=1, E=0
            [0.050, 0.950],  # B=1, E=1
        ],
    ]),
)

# P(JohnCalls | Alarm)
phi_J = Factor(
    ("Alarm", "JohnCalls"),
    alarm_cardinalities,
    np.array([
        [0.95, 0.05],
        [0.10, 0.90],
    ]),
)

# P(MaryCalls | Alarm)
phi_M = Factor(
    ("Alarm", "MaryCalls"),
    alarm_cardinalities,
    np.array([
        [0.99, 0.01],
        [0.30, 0.70],
    ]),
)

alarm_factors = [phi_B, phi_E, phi_A, phi_J, phi_M]


In [25]:
burglary_posterior = variable_elimination(
    factors=alarm_factors,
    query_variables=["Burglary"],
    evidence={
        "JohnCalls": 1,
        "MaryCalls": 1,
    },
    elimination_order=[
        "Earthquake",
        "Alarm",
    ],
)

burglary_posterior


Factor(variables=('Burglary',), shape=(2,))
[0.71582816 0.28417184]

The posterior probability of burglary should be much larger than its prior probability \(0.001\), because two observations that are likely when the alarm rings have been observed.

This is Bayesian reasoning:

- begin with a prior belief,
- receive evidence,
- propagate the evidence through the network,
- obtain an updated posterior belief.

# Inspecting one elimination step

The following cell lets us observe what happens when `Earthquake` is eliminated.

restricted = apply_evidence(
    alarm_factors,
    {
        "JohnCalls": 1,
        "MaryCalls": 1,
    },
)

print("Factors after applying evidence:")
for factor in restricted:
    print(factor.variables, factor.values.shape)

after_earthquake = eliminate_variable(
    restricted,
    "Earthquake",
)

print("\nFactors after eliminating Earthquake:")
for factor in after_earthquake:
    print(factor.variables, factor.values.shape)


## What should we notice?

Before eliminating `Earthquake`, the factors involving it are:

- $P(E)$
- $P(A\mid B,E)$

They are replaced by

$$\tau(B,A)=
\sum_E P(E)P(A\mid B,E).
$$

The variable `Earthquake` disappears, while its probabilistic influence is preserved in the new factor $tau(B,A)$.


# Why elimination order matters

Different valid elimination orders return the same final probability, but they may create intermediate factors of very different sizes.

For example:

```text
Eliminate X first:
    may create factor over A, B, C, D

Eliminate Y first:
    may create factor only over A, B
```

The numerical answer is unchanged, but the computational cost can change significantly.

For now, we will provide the order manually. Later, we can implement heuristics such as:

- minimum degree,
- minimum fill,
- weighted minimum fill.


# Summary

Variable elimination answers a probability query without constructing the complete joint distribution.

For every hidden variable:

```text
collect relevant factors
        ↓
multiply them
        ↓
sum out the hidden variable
        ↓
return the smaller factor
```

The final workflow is:

```text
factors from Bayesian network
        ↓
apply evidence
        ↓
eliminate hidden variables
        ↓
multiply remaining factors
        ↓
normalize
        ↓
posterior distribution
```

The most important conceptual statement is:

> Elimination removes a variable from the representation, not its influence from the probability calculation.
